In [1]:
import os
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

BASE_DIR = r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge"
CSV_PATH = os.path.join(BASE_DIR, "df_subset_500.csv")
IMG_DIR  = os.path.join(BASE_DIR, "imagenes")

Device: cuda


PRUEBA DE MLFLOW

In [2]:
import mlflow

mlflow.set_experiment("torax-cnn-liviana")

C:\Users\trodr\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='file:///c:/Users/trodr/Documents/proyecto-torax-v2.0/03-notebook/mlruns/1', creation_time=1781120329102, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781120329102, lifecycle_stage='active', name='torax-cnn-liviana', tags={}, trace_location=None, workspace='default'>

In [3]:
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())
print("\nPrimeras filas:")
df.head()

Shape: (530, 20)

Columnas:
['dicom_id', 'subject_id', 'study_id', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices', 'full_path', 'ViewPosition', 'calidad-imagen']

Primeras filas:


,dicom_id,subject_id,study_id,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices,full_path,ViewPosition,calidad-imagen
0,1621b732-f77b9463-87769d53-3175e5ed-1822d35f,10516278,53994171,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0,NaN,0.0,NaN,0.0,NaN,NaN,files/p10/p10516278/s53994171/1621b732-f77b946...,AP,NaN
1,a9dfa70d-5d8b5850-55eb30a9-bbceba45-50d136a6,14462350,57896727,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,files/p14/p14462350/s57896727/a9dfa70d-5d8b585...,AP,1.0
2,9fcb75dd-5d039dc4-83cb6de7-0ebcdc22-1af9fcef,19431075,59114744,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,files/p19/p19431075/s59114744/9fcb75dd-5d039dc...,AP,1.0
3,a4a85001-3068f851-a1baae43-868d1727-1bf840dc,16129520,57623500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,files/p16/p16129520/s57623500/a4a85001-3068f85...,PA,NaN
4,1c4bcdef-37e2de53-a6fe1089-216fcceb-2ac01aa6,15447983,58598537,1.0,-1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,files/p15/p15447983/s58598537/1c4bcdef-37e2de5...,AP,NaN


In [4]:
LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax"
]

# Verificar que las 14 columnas existen
faltantes = [c for c in LABEL_COLS if c not in df.columns]
print("Etiquetas faltantes:", faltantes if faltantes else "ninguna ✓")

# Revisar distribución de positivos por etiqueta (cuántos 1 hay en cada una)
print("\nPositivos por etiqueta:")
print(df[LABEL_COLS].sum().sort_values(ascending=False))

# Revisar valores únicos (deberían ser 0.0 y 1.0 tras U-zeros)
print("\nValores únicos en etiquetas:", np.unique(df[LABEL_COLS].values))

Etiquetas faltantes: ninguna ✓

Positivos por etiqueta:
No Finding                    180.0
Pleural Effusion              128.0
Lung Opacity                  109.0
Atelectasis                    98.0
Cardiomegaly                   94.0
Edema                          36.0
Pneumothorax                   26.0
Fracture                       14.0
Consolidation                  12.0
Lung Lesion                     6.0
Pleural Other                   3.0
Pneumonia                       1.0
Enlarged Cardiomediastinum     -3.0
dtype: float64

Valores únicos en etiquetas: [-1.  0.  1. nan]


In [5]:
def aplicar_u_zeros(df, label_cols):
    df = df.copy()
    df[label_cols] = df[label_cols].fillna(0.0)        # NaN -> 0
    df[label_cols] = df[label_cols].replace(-1.0, 0.0) # incierto -> 0
    return df

df = aplicar_u_zeros(df, LABEL_COLS)

print("Valores únicos tras U-zeros:", np.unique(df[LABEL_COLS].values))

Valores únicos tras U-zeros: [0. 1.]


In [6]:
# Construir full_path apuntando a 3000imagenes
df["full_path"] = df["full_path"].apply(
    lambda p: os.path.join(IMG_DIR, p.replace("/", os.sep))
)

# Verificación
existen = df["full_path"].apply(os.path.exists)
print(f"Paths que existen: {existen.sum()} / {len(df)}")
print(f"\nEjemplo: {df['full_path'].iloc[0]}")

if existen.sum() < len(df):
    print(f"\n⚠ Faltan {len(df) - existen.sum()} imágenes. Las dropeamos:")
    df = df[existen].reset_index(drop=True)
    print(f"Filas restantes: {len(df)}")

Paths que existen: 529 / 530

Ejemplo: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\imagenes\files\p10\p10516278\s53994171\1621b732-f77b9463-87769d53-3175e5ed-1822d35f.jpg

⚠ Faltan 1 imágenes. Las dropeamos:
Filas restantes: 529


In [6]:
print("Ejemplo full_path original:")
print(df["full_path"].iloc[0])
print("\nBasename (solo el nombre del archivo):", os.path.basename(df["full_path"].iloc[0]))

def rebase(df, img_dir):
    df = df.copy()
    df["img_path"] = df["full_path"].apply(
        lambda p: os.path.join(img_dir, os.path.basename(str(p)))
    )
    return df

df = rebase(df, IMG_DIR)

existen = df["img_path"].apply(os.path.exists)
print(f"\nImágenes encontradas: {existen.sum()} / {len(df)}")
if existen.sum() < len(df):
    print("\nEjemplo de ruta que NO existe:")
    print(df.loc[~existen, "img_path"].iloc[0])

Ejemplo full_path original:
files/p10/p10516278/s53994171/1621b732-f77b9463-87769d53-3175e5ed-1822d35f.jpg

Basename (solo el nombre del archivo): 1621b732-f77b9463-87769d53-3175e5ed-1822d35f.jpg

Imágenes encontradas: 0 / 530

Ejemplo de ruta que NO existe:
C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\imagenes\1621b732-f77b9463-87769d53-3175e5ed-1822d35f.jpg


In [7]:
# Reconstruir full_path apuntando a la ubicación real de las imágenes
BASE_IMAGES = r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\imagenes\files"

def build_path(row):
    sid = str(row['subject_id'])
    p_group   = f"p{sid[:2]}"      # p10, p11, ...
    p_subject = f"p{sid}"          # p10040721
    s_study   = f"s{row['study_id']}"
    dicom     = f"{row['dicom_id']}.jpg"
    return os.path.join(BASE_IMAGES, p_group, p_subject, s_study, dicom)

df['full_path'] = df.apply(build_path, axis=1)

# Verificación
existen = df['full_path'].apply(os.path.exists)
print(f"Paths que existen: {existen.sum()} / {len(df)}")
print(f"\nEjemplo: {df['full_path'].iloc[0]}")

if existen.sum() < len(df):
    print(f"\n⚠ Faltan {len(df) - existen.sum()} imágenes. Las dropeamos:")
    df = df[existen].reset_index(drop=True)
    print(f"Filas restantes: {len(df)}")

Paths que existen: 529 / 530

Ejemplo: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\imagenes\files\p10\p10516278\s53994171\1621b732-f77b9463-87769d53-3175e5ed-1822d35f.jpg

⚠ Faltan 1 imágenes. Las dropeamos:
Filas restantes: 529


In [7]:
class ToraxDataset(Dataset):
    def __init__(self, df, label_cols, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_cols = label_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. Cargar la imagen desde disco
        img = Image.open(row["full_path"]).convert("RGB")

        # 2. Aplicar transformaciones (resize, normalizar, a tensor)
        if self.transform:
            img = self.transform(img)

        # 3. Armar el vector de etiquetas como tensor float
        labels = torch.tensor(row[self.label_cols].values.astype("float32"))

        return img, labels

In [8]:
def split_patient_level(df, group_col="subject_id", test_size=0.2, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, val_idx = next(gss.split(df, groups=df[group_col]))
    return df.iloc[train_idx], df.iloc[val_idx]

train_df, val_df = split_patient_level(df)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")
solapan = set(train_df["subject_id"]) & set(val_df["subject_id"])
print("Pacientes solapados (debe ser 0):", len(solapan))

IMG_SIZE = 224
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = ToraxDataset(train_df, LABEL_COLS, train_tf)
val_ds   = ToraxDataset(val_df,   LABEL_COLS, val_tf)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=0)

imgs, lbls = next(iter(train_loader))
print("\nBatch imágenes:", imgs.shape)   # esperado [16, 3, 224, 224]
print("Batch etiquetas:", lbls.shape)    # esperado [16, 13]

Train: 422 | Val: 107
Pacientes solapados (debe ser 0): 0

Batch imágenes: torch.Size([16, 3, 224, 224])
Batch etiquetas: torch.Size([16, 13])


In [9]:
class ToraxCNN(nn.Module):
    def __init__(self, n_classes=13):
        super().__init__()

        # --- Extractor de características ---
        self.features = nn.Sequential(
            # Bloque 1
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),                # 224 -> 112
            # Bloque 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                # 112 -> 56
            # Bloque 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                # 56 -> 28
            # Bloque 4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),                # 28 -> 14
        )

        # --- Clasificador ---
        self.pool = nn.AdaptiveAvgPool2d(1)   # [128, 14, 14] -> [128, 1, 1]
        self.classifier = nn.Sequential(
            nn.Flatten(),                     # [128, 1, 1] -> [128]
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),        # [128] -> [13]
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

model = ToraxCNN(n_classes=len(LABEL_COLS)).to(DEVICE)

# Verificar shapes con un tensor de prueba
x_prueba = torch.randn(2, 3, 224, 224).to(DEVICE)
salida = model(x_prueba)
print("Entrada:", x_prueba.shape)   # [2, 3, 224, 224]
print("Salida:", salida.shape)      # [2, 13]

Entrada: torch.Size([2, 3, 224, 224])
Salida: torch.Size([2, 13])


In [10]:
# Calcular pos_weight desde el train set
train_labels = torch.tensor(train_df[LABEL_COLS].values, dtype=torch.float32)
n_pos = train_labels.sum(dim=0).clamp(min=1)
n_neg = (train_labels == 0).sum(dim=0).clamp(min=1)
pos_weight = (n_neg / n_pos).to(DEVICE)

print("pos_weight por label:")
for name, w in zip(LABEL_COLS, pos_weight.cpu()):
    print(f"  {name:<30} {w:.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

# Verificación con un batch real
imgs, lbls = next(iter(train_loader))
imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
logits = model(imgs)
loss = criterion(logits, lbls)
print("\nLogits shape:", logits.shape)
print("Labels shape:", lbls.shape)
print("Pérdida inicial:", loss.item())

pos_weight por label:
  Atelectasis                    3.22
  Cardiomegaly                   3.91
  Consolidation                  23.82
  Edema                          6.03
  Enlarged Cardiomediastinum     29.14
  Fracture                       29.14
  Lung Lesion                    45.89
  Lung Opacity                   3.35
  No Finding                     2.10
  Pleural Effusion               2.70
  Pleural Other                  139.67
  Pneumonia                      11.79
  Pneumothorax                   21.21

Logits shape: torch.Size([16, 13])
Labels shape: torch.Size([16, 13])
Pérdida inicial: 1.1354063749313354


In [11]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()                      # modo entrenamiento (activa dropout, etc.)
    loss_total = 0.0

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)

        optimizer.zero_grad()          # 1. limpiar gradientes viejos
        logits = model(imgs)           # 2. forward: predicción
        loss = criterion(logits, lbls) # 3. calcular el error
        loss.backward()                # 4. backward: calcular gradientes
        optimizer.step()               # 5. actualizar los pesos

        loss_total += loss.item()

    return loss_total / len(loader)    # pérdida promedio de la época

In [12]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

def evaluate(model, loader, criterion, device, label_cols, umbral=0.5):
    model.eval()                       # modo evaluación (apaga dropout)
    loss_total = 0.0
    todas_probs, todas_lbls = [], []

    with torch.no_grad():              # no calcular gradientes
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss_total += criterion(logits, lbls).item()

            probs = torch.sigmoid(logits)   # logits -> probabilidades
            todas_probs.append(probs.cpu())
            todas_lbls.append(lbls.cpu())

    # Juntar todos los batches
    y_prob = torch.cat(todas_probs).numpy()
    y_true = torch.cat(todas_lbls).numpy()
    y_pred = (y_prob >= umbral).astype(int)   # binarizar con el umbral

    # AUC por etiqueta (independiente del umbral)
    aucs = {}
    for i, nombre in enumerate(label_cols):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        aucs[nombre] = roc_auc_score(y_true[:, i], y_prob[:, i])

    # Precision, recall, F1 macro (dependen del umbral)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
        # Por etiqueta para el desglose
    prec_arr, rec_arr, f1_arr, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    
        
    # TP, FP, FN, TN por etiqueta
    confusion_por_etiqueta = {}
    for i, nombre in enumerate(label_cols):
        tn, fp, fn, tp = confusion_matrix(
            y_true[:, i], y_pred[:, i], labels=[0, 1]
        ).ravel()
        confusion_por_etiqueta[nombre] = {
            "TP": int(tp), "FP": int(fp),
            "FN": int(fn), "TN": int(tn)
        }
    # Métricas por etiqueta consolidadas
    metricas_por_etiqueta = {}
    for i, nombre in enumerate(label_cols):
        metricas_por_etiqueta[nombre] = {
            "auc":       aucs.get(nombre, float("nan")),
            "precision": float(prec_arr[i]),
            "recall":    float(rec_arr[i]),
            "f1":        float(f1_arr[i]),
            **confusion_por_etiqueta[nombre],   # TP, FP, FN, TN
        }


    metricas = {
        "loss": loss_total / len(loader),
        "auc": np.mean(list(aucs.values())) if aucs else float("nan"),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    return metricas, aucs, metricas_por_etiqueta   # <- ahora son DOS valores

In [13]:
import inspect
print("loss" in inspect.getsource(evaluate))

True


In [14]:
# Correr ANTES del loop MLflow
model = ToraxCNN(n_classes=len(LABEL_COLS)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [16]:
NUM_EPOCHS = 10
mejor_auc = 0.0
mejor_epoch = 0
with mlflow.start_run(run_name="cnn-liviana3000"):

    # --- Registrar los parámetros del experimento (una sola vez) ---
    mlflow.log_params({
        "modelo": "ToraxCNN_liviana",
        "condicion": "sin_limpieza",      # <- tu variable independiente
        "dataset": "df_subset_500",
        "n_etiquetas": len(LABEL_COLS),
        "n_train": len(train_df),
        "n_val": len(val_df),
        "lr": 1e-3,
        "batch_size": 16,
        "num_epochs": NUM_EPOCHS,
        "img_size": IMG_SIZE,
        "umbral": 0.5,
    })


    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        metricas, aucs_detalle, metricas_por_etiqueta = evaluate(model, val_loader, criterion, DEVICE, LABEL_COLS)

        
        print(f"Época {epoch:2d}/{NUM_EPOCHS} | "
            f"train_loss: {train_loss:.4f} | "
            f"val_loss: {metricas['loss']:.4f} | "
            f"val_AUC: {metricas['auc']:.4f}")

        mlflow.log_metrics({
                "train_loss": train_loss,
                "val_loss": metricas["loss"],
                "val_auc": metricas["auc"],
                "val_precision": metricas["precision"],
                "val_recall": metricas["recall"],
                "val_f1": metricas["f1"],
            }, step=epoch)


        if metricas["auc"] > mejor_auc:
            mejor_auc = metricas["auc"]
            mejor_epoch = epoch
            torch.save(model.state_dict(), "mejor_modelo.pth")
            print(f"   ↑ nuevo mejor AUC: {mejor_auc:.4f} (época {mejor_epoch})")
        scheduler.step(metricas["auc"])
            
    model.load_state_dict(torch.load("mejor_modelo.pth"))
    _, aucs_final, metricas_por_etiqueta_final = evaluate(
        model, val_loader, criterion, DEVICE, LABEL_COLS
    )

        # AUC por etiqueta
    for label, auc_val in aucs_final.items():
        if not np.isnan(auc_val):
            mlflow.log_metric(f"auc_{label}", auc_val)

    # Precision, Recall, F1, TP, FP, FN, TN por etiqueta
    for label, vals in metricas_por_etiqueta_final.items():
        mlflow.log_metrics({
            f"precision_{label}": vals["precision"],
            f"recall_{label}":    vals["recall"],
            f"f1_{label}":        vals["f1"],
            f"TP_{label}":        vals["TP"],
            f"FP_{label}":        vals["FP"],
            f"FN_{label}":        vals["FN"],
            f"TN_{label}":        vals["TN"],
        })

    # Resumen del run
    mlflow.log_metrics({
        "mejor_val_auc": mejor_auc,
        "mejor_epoch":   mejor_epoch,
    })

    # CSV artifact consolidado
    df_por_etiqueta = pd.DataFrame(metricas_por_etiqueta_final).T
    df_por_etiqueta.to_csv("metricas_por_etiqueta.csv")
    mlflow.log_artifact("metricas_por_etiqueta.csv")
    mlflow.log_artifact("mejor_modelo.pth")

print(f"\nListo. Mejor AUC: {mejor_auc:.4f} en época {mejor_epoch}")

Época  1/10 | train_loss: 1.2959 | val_loss: 0.9973 | val_AUC: 0.4854
   ↑ nuevo mejor AUC: 0.4854 (época 1)
Época  2/10 | train_loss: 1.2128 | val_loss: 1.0078 | val_AUC: 0.6518
   ↑ nuevo mejor AUC: 0.6518 (época 2)
Época  3/10 | train_loss: 1.2257 | val_loss: 1.0146 | val_AUC: 0.6802
   ↑ nuevo mejor AUC: 0.6802 (época 3)
Época  4/10 | train_loss: 1.1630 | val_loss: 1.0117 | val_AUC: 0.6730
Época  5/10 | train_loss: 1.1684 | val_loss: 1.0594 | val_AUC: 0.5127
Época  6/10 | train_loss: 1.1831 | val_loss: 0.9973 | val_AUC: 0.6334
Época  7/10 | train_loss: 1.1590 | val_loss: 1.0087 | val_AUC: 0.6543
Época  8/10 | train_loss: 1.1576 | val_loss: 0.9814 | val_AUC: 0.6310
Época  9/10 | train_loss: 1.1320 | val_loss: 0.9913 | val_AUC: 0.7302
   ↑ nuevo mejor AUC: 0.7302 (época 9)
Época 10/10 | train_loss: 1.1290 | val_loss: 0.9644 | val_AUC: 0.7293

Listo. Mejor AUC: 0.7302 en época 9


In [17]:
# --- Print final del run ---
print(f"\n{'='*60}")
print(f"  RESULTADOS FINALES — {mejor_epoch} épocas")
print(f"{'='*60}")
print(f"  Mejor época:  {mejor_epoch}")
print(f"  Mejor AUC:    {mejor_auc:.4f}")
print(f"{'='*60}")
print(f"\n{'Etiqueta':<30} {'AUC':>6} {'Prec':>6} {'Recall':>6} {'F1':>6} {'TP':>4} {'FP':>4} {'FN':>4} {'TN':>4}")
print(f"{'-'*74}")
for label, vals in metricas_por_etiqueta_final.items():
    auc_str = f"{vals['auc']:.4f}" if not np.isnan(vals['auc']) else "  nan"
    print(f"  {label:<28} {auc_str:>6} {vals['precision']:>6.4f} {vals['recall']:>6.4f} {vals['f1']:>6.4f} {vals['TP']:>4} {vals['FP']:>4} {vals['FN']:>4} {vals['TN']:>4}")
print(f"{'='*60}\n")


  RESULTADOS FINALES — 9 épocas
  Mejor época:  9
  Mejor AUC:    0.7302

Etiqueta                          AUC   Prec Recall     F1   TP   FP   FN   TN
--------------------------------------------------------------------------
  Atelectasis                  0.7337 0.3636 0.1818 0.2424    4    7   18   78
  Cardiomegaly                 0.6776 0.1429 0.0526 0.0769    1    6   18   82
  Consolidation                0.6571 0.0339 0.6667 0.0645    2   57    1   47
  Edema                        0.8097 0.2500 0.0909 0.1333    1    3   10   93
  Enlarged Cardiomediastinum   0.8491 0.0000 0.0000 0.0000    0   10    1   96
  Fracture                     0.9906 0.0172 1.0000 0.0339    1   57    0   49
  Lung Lesion                  0.8679 0.0099 1.0000 0.0196    1  100    0    6
  Lung Opacity                 0.5321 0.1739 0.1905 0.1818    4   19   17   67
  No Finding                   0.6807 0.4828 0.9545 0.6412   42   45    2   18
  Pleural Effusion             0.7631 0.2500 0.0417 0.0714  

In [ ]:
import pandas as pd
print(pd.Series(aucs_detalle).sort_values(ascending=False))

Enlarged Cardiomediastinum    0.915094
Pneumothorax                  0.835859
Consolidation                 0.830128
Edema                         0.807765
Pleural Effusion              0.799197
No Finding                    0.753247
Cardiomegaly                  0.742225
Atelectasis                   0.705348
Lung Opacity                  0.684385
Pneumonia                     0.509615
Lung Lesion                   0.481132
Fracture                      0.009434
dtype: float64


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def evaluate(model, loader, criterion, device, label_cols, umbral=0.5):
    model.eval()
    loss_total = 0.0
    todas_probs, todas_lbls = [], []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss_total += criterion(logits, lbls).item()
            todas_probs.append(torch.sigmoid(logits).cpu())
            todas_lbls.append(lbls.cpu())

    y_prob = torch.cat(todas_probs).numpy()
    y_true = torch.cat(todas_lbls).numpy()
    y_pred = (y_prob >= umbral).astype(int)   # binarizar con el umbral

    # AUC por etiqueta (independiente del umbral)
    aucs = {}
    for i, nombre in enumerate(label_cols):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        aucs[nombre] = roc_auc_score(y_true[:, i], y_prob[:, i])

    # Precision, recall, F1 macro (dependen del umbral)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    metricas = {
        "loss": loss_total / len(loader),
        "auc": np.mean(list(aucs.values())) if aucs else float("nan"),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    return metricas, aucs

In [ ]:
NUM_EPOCHS = 10
mejor_auc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    metricas, aucs_detalle = evaluate(model, val_loader, criterion, DEVICE, LABEL_COLS)

    print(f"Época {epoch:2d}/{NUM_EPOCHS} | "
          f"train_loss: {train_loss:.4f} | "
          f"val_loss: {metricas['loss']:.4f} | "
          f"AUC: {metricas['auc']:.4f} | "
          f"P: {metricas['precision']:.4f} | "
          f"R: {metricas['recall']:.4f} | "
          f"F1: {metricas['f1']:.4f}")

    if metricas["auc"] > mejor_auc:
        mejor_auc = metricas["auc"]
        torch.save(model.state_dict(), "mejor_modelo.pth")
        print(f"   ↑ nuevo mejor AUC: {mejor_auc:.4f} (modelo guardado)")

print(f"\nListo. Mejor AUC de validación: {mejor_auc:.4f}")

Época  1/10 | train_loss: 0.3123 | val_loss: 0.2948 | AUC: 0.6299 | P: 0.0349 | R: 0.0682 | F1: 0.0462
   ↑ nuevo mejor AUC: 0.6299 (modelo guardado)
Época  2/10 | train_loss: 0.3092 | val_loss: 0.2710 | AUC: 0.6946 | P: 0.0604 | R: 0.0192 | F1: 0.0292
   ↑ nuevo mejor AUC: 0.6946 (modelo guardado)
Época  3/10 | train_loss: 0.3037 | val_loss: 0.2618 | AUC: 0.6828 | P: 0.0955 | R: 0.0533 | F1: 0.0623
Época  4/10 | train_loss: 0.3044 | val_loss: 0.2658 | AUC: 0.6749 | P: 0.0469 | R: 0.0437 | F1: 0.0452
Época  5/10 | train_loss: 0.2971 | val_loss: 0.2678 | AUC: 0.6632 | P: 0.2150 | R: 0.0670 | F1: 0.0922
Época  6/10 | train_loss: 0.3058 | val_loss: 0.2647 | AUC: 0.6766 | P: 0.1017 | R: 0.0589 | F1: 0.0742
Época  7/10 | train_loss: 0.3035 | val_loss: 0.2728 | AUC: 0.6330 | P: 0.0429 | R: 0.0507 | F1: 0.0465
Época  8/10 | train_loss: 0.3095 | val_loss: 0.2739 | AUC: 0.6822 | P: 0.1458 | R: 0.0488 | F1: 0.0699
Época  9/10 | train_loss: 0.3030 | val_loss: 0.2656 | AUC: 0.7009 | P: 0.0520 | R: